In [0]:
import pyspark
sales_df=spark.read.option('header','true').option('inferSchema','true').csv('/Volumes/workspace/ds_27/batch_27/sales.csv')
products_df = spark.read.option("header",True).csv("/Volumes/workspace/ds_27/batch_27/products.csv")

stores_df = spark.read.option("header",True).csv("/Volumes/workspace/ds_27/batch_27/stores.csv")


In [0]:
sales_df.dropna()

In [0]:
final_df=sales_df.join(products_df,'product_id').join(stores_df,'store_id')
display(final_df)

In [0]:
final_df.createOrReplaceTempView("sales_data")

In [0]:
from pyspark.sql.functions import col, when
from pyspark import sql

result = spark.sql("""
SELECT
category,
SUM(sales_amount) as total_revenue
FROM sales_data
GROUP BY category
ORDER BY total_revenue DESC
""")

display(result)

In [0]:
top_selling_product=spark.sql("""
SELECT
product_name,
SUM(quantity) total_qty
FROM sales_data
GROUP BY product_name
ORDER BY total_qty DESC
""")
display(top_selling_product)


In [0]:
final_df=final_df.withColumn(
    "sales_type",
    when(col("sales_amount")>150,"High")
    .otherwise("low")
)    

In [0]:
final_df.write.mode("overwrite").option("header",True).csv("/Volumes/workspace/ds_27/batch_27/final_df")
final_df.show()